# Stage 2 event-first experiment gate

우선 collision frame만 개선한다. entry/evasion/side를 같은 첫 실험에 섞지 않는다. 이 노트북은 사고 clip의 시간 라벨 품질과 event-holdout을 검증하는 설계다.

In [ ]:
from pathlib import Path
import pandas as pd

DRIVE_ROOT = Path('/content/drive/MyDrive/블랙박스 영상 기반 사고 분석')
PROJECT_ROOT = Path('/content/dacon236753')
EXPERIMENT = 's2_event_v1'
DATA_ROOT = DRIVE_ROOT / 'external_data' / 'stage2'
MANIFEST = DATA_ROOT / 'collision_manifest.csv'
ARTIFACT = DRIVE_ROOT / 'experiments' / EXPERIMENT
for folder in ('config','checkpoints','predictions','reports','submission','logs'):
    (ARTIFACT / folder).mkdir(parents=True, exist_ok=True)
print('Expected:', MANIFEST)
print('Experiment:', ARTIFACT)

## collision manifest 계약

필수 열: `id,source_group,video_path,fps,collision_time_seconds`. 선택 열: `entry_time_seconds,evasion_space,entry_side,license_ref`.

- 사고 time은 실제 접촉 frame만 허용한다. 위험 감지/근접 시점은 별도 label로 저장한다.
- source_group는 원본 YouTube/Nexar sequence 수준으로 분리한다.
- CCD/DADA-2000 등 외부 label은 정의를 대회 정의와 비교한 뒤에만 병합한다.

In [ ]:
REQUIRED = {'id','source_group','video_path','fps','collision_time_seconds'}
if not MANIFEST.exists():
    print('STOP: 충돌 manifest와 라이선스 기록이 필요합니다.')
else:
    data = pd.read_csv(MANIFEST)
    assert not (REQUIRED - set(data.columns)), '필수 열 누락'
    assert data.id.is_unique and (data.fps > 0).all()
    assert (data.collision_time_seconds >= 0).all()
    print(f'PASS: {len(data)} clips / {data.source_group.nunique()} source groups')

## V1 모델과 판단

1. vehicle detector/tracker + global motion으로 후보 frame top-K 생성.
2. 후보 ±2초 clip을 temporal event head로 재순위화.
3. 예측값을 실제 원본 frame filename으로 snap.
4. group-holdout Accuracy@0.3s가 기준선보다 개선될 때만 제출 후보.

그 다음에만 피해차량 track→차선 경계 침범으로 entry를 만들고, 수동 정밀 라벨 150–300개를 확보해 evasion/side head를 추가한다.

In [ ]:
# FINAL SUBMISSION GATE — 이 셀은 항상 마지막에 둡니다.
import subprocess, sys
BASELINE_LOCAL = ARTIFACT / 'reports/baseline_local_metrics.json'
CANDIDATE_LOCAL = ARTIFACT / 'reports/local_metrics.json'
ZIP_SMOKE_MARKER = ARTIFACT / 'reports/zip_smoke_pass.json'
gate = [sys.executable, str(PROJECT_ROOT / 'tools/go_no_go.py'), '--stage', 'stage2', '--baseline-local', str(BASELINE_LOCAL), '--candidate-local', str(CANDIDATE_LOCAL), '--evaluation-kind', 'external_group_holdout', '--json-out', str(ARTIFACT / 'reports/submission_gate.json')]
if ZIP_SMOKE_MARKER.exists(): gate.append('--smoke-pass')
subprocess.run(gate, check=False)
